<a href="https://colab.research.google.com/github/lucasmsorrentino/Git_GenAI/blob/main/Parte_2_Funcional_Trocando_a_cabe%C3%A7a_de_Classifica%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Só se precisar
#!pip install -q -U transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 16.4 MB/s eta 0:00:00


In [1]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")

Hugging Face token: ··········


In [2]:
import torch
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer carregado")
print("Vocabulário:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Tokenizer carregado
Vocabulário: 128000


### Padding

Para formar um batch, precisamos que as sequências tenham o mesmo tamanho. O Llama não possui originalmente um token de padding separado, então vamos usar o token de fim de texto como padding.

In [3]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

PAD token: <|end_of_text|>
PAD token ID: 128001


In [4]:
text = "This movie is absolutely fantastic!"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer(text)["input_ids"]

print("Texto:")
print(text)
print("\nTokens:")
print(tokens)
print("\nIDs:")
print(token_ids)

Texto:
This movie is absolutely fantastic!

Tokens:
['This', 'Ġmovie', 'Ġis', 'Ġabsolutely', 'Ġfantastic', '!']

IDs:
[128000, 2028, 5818, 374, 11112, 14964, 0]


In [5]:
encoded = tokenizer(text)
decoded = tokenizer.decode(encoded["input_ids"])

print("Original:", text)
print("Decodificado:", decoded)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Original: This movie is absolutely fantastic!
Decodificado: <|begin_of_text|>This movie is absolutely fantastic!


## 5. Carregando o modelo para classificação

O Llama original é um modelo causal de linguagem. Sua saída original é voltada para prever o próximo token.

Agora vamos usar a mesma arquitetura Transformer, mas com uma **cabeça de classificação** que produz apenas duas saídas:

```text
0 → negative
1 → positive
```

A camada de classificação é inicializada para esta nova tarefa e será treinada junto com o restante do Llama.

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    dtype=torch.bfloat16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id

print("Modelo carregado")

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-1B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo carregado


## Conhecendo o modelo

In [7]:
print("Hidden size:", model.config.hidden_size)
print("Número de camadas:", model.config.num_hidden_layers)
print("Número de labels:", model.config.num_labels)
print("Tipo do modelo:", type(model).__name__)

Hidden size: 2048
Número de camadas: 16
Número de labels: 2
Tipo do modelo: LlamaForSequenceClassification


### Quantos parâmetros serão treinados?

In [8]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Parâmetros totais:      {total_params:,}")
print(f"Parâmetros treináveis:  {trainable_params:,}")

Parâmetros totais:      1,235,818,496
Parâmetros treináveis:  1,235,818,496


In [9]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/sst2")

train = dataset["train"].shuffle().select(range(1000))
validation = dataset["validation"]

print("Treinamento:", len(train))
print("Validação:", len(validation))

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Treinamento: 1000
Validação: 872


## Observando os dados de treinamento

In [10]:
label_names = {
    0: "negative",
    1: "positive"
}

for i in range(10):
    print(f"Exemplo {i + 1}")
    print("Texto:", train[i]["sentence"])
    print("Label:", train[i]["label"], "(", label_names[train[i]["label"]], ")")
    print()

Exemplo 1
Texto: there surrender $ 9 
Label: 0 ( negative )

Exemplo 2
Texto: , two towers outdoes its spectacle . 
Label: 1 ( positive )

Exemplo 3
Texto: extensive post-production 
Label: 1 ( positive )

Exemplo 4
Texto: picture that at the very least has a spark of life to it -- more than you can say for plenty of movies that flow through the hollywood pipeline without a hitch 
Label: 1 ( positive )

Exemplo 5
Texto: by poor casting 
Label: 0 ( negative )

Exemplo 6
Texto: emotionally manipulative and sadly imitative of innumerable past love story derisions 
Label: 0 ( negative )

Exemplo 7
Texto: downright hitchcockian 
Label: 1 ( positive )

Exemplo 8
Texto: sweet gentle jesus 
Label: 1 ( positive )

Exemplo 9
Texto: strong and convincing 
Label: 1 ( positive )

Exemplo 10
Texto: too many characters saying too many clever things and getting into too many pointless situations 
Label: 0 ( negative )



## 9. Tokenizando o dataset

Agora transformamos cada frase em `input_ids` e `attention_mask`.

Vamos limitar as frases a 128 tokens.

In [11]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train = train.map(tokenize, batched=True)
validation = validation.map(tokenize, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [12]:
print(train.column_names)
print("\nPrimeiro exemplo tokenizado:")
print(train[0])

['idx', 'sentence', 'label', 'input_ids', 'attention_mask']

Primeiro exemplo tokenizado:
{'idx': 48373, 'sentence': 'there surrender $ 9 ', 'label': 0, 'input_ids': [128000, 19041, 32859, 400, 220, 24, 220, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 

In [13]:
train.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

validation.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [14]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [15]:
def predict(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)
    predictions = probabilities.argmax(dim=1)

    return predictions, probabilities


examples = [
    "This movie is fantastic!",
    "This movie is terrible!",
    "I really enjoyed this film.",
    "I hated this film."
]

predictions, probabilities = predict(examples)

for text, prediction, probability in zip(
    examples, predictions, probabilities
):
    print(f"{label_names[prediction.item()]:8s} {probability[prediction].item():.3f}  {text}")

negative 0.945  This movie is fantastic!
negative 0.840  This movie is terrible!
negative 0.930  I really enjoyed this film.
negative 0.961  I hated this film.


**Não esperamos bons resultados ainda.** A cabeça de classificação ainda não foi treinada para reconhecer sentimentos.

In [16]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [17]:
NUM_EPOCHS = 2

history = []

for epoch in range(NUM_EPOCHS):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["label"].to(model.device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
                f"batch {batch_idx + 1}/{len(train_loader)} | "
                f"loss {loss.item():.4f}"
            )

    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total

    history.append({
        "loss": epoch_loss,
        "accuracy": epoch_accuracy
    })

    print(
        f"\nEpoch {epoch + 1}: "
        f"loss = {epoch_loss:.4f}, "
        f"accuracy = {epoch_accuracy:.4f}\n"
    )

Epoch 1/2 | batch 100/125 | loss 0.0903

Epoch 1: loss = 0.4051, accuracy = 0.8480

Epoch 2/2 | batch 100/125 | loss 0.0227

Epoch 2: loss = 0.0566, accuracy = 0.9810



## Avaliando no conjunto de validação

Cuidado. Vamos usar a validação como se fosse um teste. O conjunto de testes do SST2 não possui labels.

In [18]:
model.eval()

correct = 0
total = 0
validation_loss = 0.0

with torch.no_grad():
    for batch in validation_loader:

        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["label"].to(model.device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        validation_loss += outputs.loss.item()

        predictions = outputs.logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

validation_loss /= len(validation_loader)
validation_accuracy = correct / total

print(f"Validation loss:     {validation_loss:.4f}")
print(f"Validation accuracy: {validation_accuracy:.4f}")

Validation loss:     0.2537
Validation accuracy: 0.9381


## Testando frases novas

In [22]:
examples = [
    "This movie was absolutely fantastic.",
    "This movie was terrible.",
    "I really enjoyed this film.",
    "The story was boring and predictable.",
    "One of the best movies I have ever seen.",
    "I regret watching this movie.",
    "Well, that went swimmingly.",
    "That could not have gone any better.",
    "Nailed it.",
    "10/10, would recommend.",
    "That was a regular human masterclass.",
    "The design is very human.",
    "They cooked. Nobody ate, but they cooked.",
    "It’s giving 'graphic design is my passion'."


]

predictions, probabilities = predict(examples)

for text, prediction, probability in zip(
    examples, predictions, probabilities
):
    sentiment = label_names[prediction.item()]
    confidence = probability[prediction].item()

    print(f"{sentiment:8s} {confidence:.3f}  {text}")

positive 0.957  This movie was absolutely fantastic.
negative 0.992  This movie was terrible.
positive 0.988  I really enjoyed this film.
negative 0.977  The story was boring and predictable.
positive 0.953  One of the best movies I have ever seen.
negative 0.992  I regret watching this movie.
positive 0.637  Well, that went swimmingly.
positive 0.895  That could not have gone any better.
positive 0.953  Nailed it.
positive 0.879  10/10, would recommend.
positive 0.906  That was a regular human masterclass.
positive 0.977  The design is very human.
negative 0.531  They cooked. Nobody ate, but they cooked.
positive 0.922  It’s giving 'graphic design is my passion'.
